# Лабораторна робота №4

## Візуалізація даних 2

**Студент:** Сапронов Анатолій  
**Група:** ФБ-45  
**Дисципліна:** Засоби підготовки та аналізу даних

## Мета роботи

Метою роботи є створення інтерактивної програми для візуалізації гармонічної функції з шумом, зміни параметрів сигналу за допомогою елементів керування та фільтрації шуму.


## 1. Постановка задачі

Потрібно створити програму, яка будує графік гармоніки:

```text
y(t) = A * sin(ωt + φ)
```

де:
- `A` — амплітуда;
- `ω` — частота;
- `φ` — фазовий зсув.

До гармоніки додається шум, який задається середнім значенням та дисперсією.  
Користувач має змогу змінювати параметри за допомогою інтерактивних елементів.


## 2. Імпорт бібліотек

У роботі використовуються:
- `numpy` — для обчислень;
- `matplotlib` — для побудови графіків;
- `matplotlib.widgets` — для створення слайдерів, кнопки та чекбокса;
- `scipy.signal` — для фільтрації сигналу.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.widgets import Slider, Button, CheckButtons
from scipy.signal import butter, filtfilt


## 3. Функції для побудови гармоніки, шуму та фільтрації

У цьому блоці реалізовано:
- функцію для створення чистої гармоніки;
- функцію для створення шуму;
- функцію для додавання шуму до гармоніки;
- функцію фільтрації зашумленого сигналу.


In [ ]:
def harmonic(t, amplitude, frequency, phase):
    """Створює чисту гармоніку y(t) = A * sin(wt + phase)."""
    return amplitude * np.sin(2 * np.pi * frequency * t + phase)


def generate_noise(size, noise_mean, noise_covariance, seed=None):
    """Створює нормальний шум із заданим середнім та дисперсією."""
    rng = np.random.default_rng(seed)
    noise_std = np.sqrt(noise_covariance)
    return rng.normal(noise_mean, noise_std, size)


def add_noise(clean_signal, noise):
    """Додає шум до чистої гармоніки."""
    return clean_signal + noise


def filter_signal(noisy_signal, cutoff_frequency, sampling_frequency, order=4):
    """Фільтрує сигнал низькочастотним фільтром Butterworth."""
    nyquist_frequency = 0.5 * sampling_frequency
    normalized_cutoff = cutoff_frequency / nyquist_frequency

    # Захист від некоректних значень слайдера
    normalized_cutoff = min(max(normalized_cutoff, 0.01), 0.99)

    b, a = butter(order, normalized_cutoff, btype="low", analog=False)
    filtered = filtfilt(b, a, noisy_signal)
    return filtered


## 4. Початкові параметри

Початкові значення потрібні для запуску програми та роботи кнопки `Reset`.


In [ ]:
initial_amplitude = 1.0
initial_frequency = 0.3
initial_phase = 0.0
initial_noise_mean = 0.0
initial_noise_covariance = 0.1
initial_cutoff_frequency = 1.0
initial_show_noise = True

t_start = 0
t_end = 10
number_of_points = 1000

t = np.linspace(t_start, t_end, number_of_points)
sampling_frequency = number_of_points / (t_end - t_start)

# Фіксований шум на старті.
# За умовою, якщо змінюються параметри гармоніки, але не шуму,
# шум не має генеруватися заново.
current_noise = generate_noise(
    size=len(t),
    noise_mean=initial_noise_mean,
    noise_covariance=initial_noise_covariance,
    seed=42
)

clean_y = harmonic(t, initial_amplitude, initial_frequency, initial_phase)
noisy_y = add_noise(clean_y, current_noise)
filtered_y = filter_signal(noisy_y, initial_cutoff_frequency, sampling_frequency)


## 5. Інтерактивна програма

Нижче створено інтерактивний графік з такими елементами:
- поле графіка;
- слайдер амплітуди;
- слайдер частоти;
- слайдер фази;
- слайдер середнього значення шуму;
- слайдер дисперсії шуму;
- слайдер частоти зрізу фільтра;
- чекбокс `Show Noise`;
- кнопка `Reset`.

> Якщо в Jupyter Notebook слайдери не відображаються інтерактивно, потрібно запустити notebook локально або використати `%matplotlib widget`.


In [ ]:
# Для локального Jupyter Notebook можна розкоментувати цей рядок:
# %matplotlib widget

fig, ax = plt.subplots(figsize=(10, 6))
plt.subplots_adjust(left=0.1, bottom=0.42)

clean_line, = ax.plot(t, clean_y, linewidth=2, label="Clean harmonic")
noisy_line, = ax.plot(t, noisy_y, alpha=0.6, label="Noisy harmonic")
filtered_line, = ax.plot(t, filtered_y, linewidth=2, label="Filtered harmonic")

ax.set_title("Гармоніка з шумом та фільтрацією")
ax.set_xlabel("t")
ax.set_ylabel("y(t)")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")

# Осі для слайдерів
ax_amplitude = plt.axes([0.18, 0.31, 0.65, 0.03])
ax_frequency = plt.axes([0.18, 0.27, 0.65, 0.03])
ax_phase = plt.axes([0.18, 0.23, 0.65, 0.03])
ax_noise_mean = plt.axes([0.18, 0.19, 0.65, 0.03])
ax_noise_covariance = plt.axes([0.18, 0.15, 0.65, 0.03])
ax_cutoff = plt.axes([0.18, 0.11, 0.65, 0.03])

amplitude_slider = Slider(ax_amplitude, "Amplitude", 0.1, 3.0, valinit=initial_amplitude)
frequency_slider = Slider(ax_frequency, "Frequency", 0.1, 2.0, valinit=initial_frequency)
phase_slider = Slider(ax_phase, "Phase", -np.pi, np.pi, valinit=initial_phase)
noise_mean_slider = Slider(ax_noise_mean, "Noise Mean", -1.0, 1.0, valinit=initial_noise_mean)
noise_covariance_slider = Slider(ax_noise_covariance, "Noise Covariance", 0.001, 1.0, valinit=initial_noise_covariance)
cutoff_slider = Slider(ax_cutoff, "Cutoff Frequency", 0.1, 5.0, valinit=initial_cutoff_frequency)

# Кнопка Reset
reset_ax = plt.axes([0.1, 0.03, 0.12, 0.05])
reset_button = Button(reset_ax, "Reset")

# Checkbox Show Noise
checkbox_ax = plt.axes([0.78, 0.025, 0.18, 0.08])
show_noise_checkbox = CheckButtons(checkbox_ax, ["Show Noise"], [initial_show_noise])

# Змінна для збереження шуму між оновленнями
state = {
    "noise": current_noise.copy(),
    "noise_mean": initial_noise_mean,
    "noise_covariance": initial_noise_covariance,
    "show_noise": initial_show_noise
}


def update_plot(_=None):
    amplitude = amplitude_slider.val
    frequency = frequency_slider.val
    phase = phase_slider.val
    noise_mean = noise_mean_slider.val
    noise_covariance = noise_covariance_slider.val
    cutoff_frequency = cutoff_slider.val

    # Якщо змінили параметри шуму — генеруємо новий шум.
    # Якщо змінювали тільки параметри гармоніки — шум залишається таким самим.
    if (
        noise_mean != state["noise_mean"] or
        noise_covariance != state["noise_covariance"]
    ):
        state["noise"] = generate_noise(
            size=len(t),
            noise_mean=noise_mean,
            noise_covariance=noise_covariance,
            seed=42
        )
        state["noise_mean"] = noise_mean
        state["noise_covariance"] = noise_covariance

    clean_signal = harmonic(t, amplitude, frequency, phase)
    noisy_signal = add_noise(clean_signal, state["noise"])
    filtered_signal = filter_signal(noisy_signal, cutoff_frequency, sampling_frequency)

    clean_line.set_ydata(clean_signal)
    noisy_line.set_ydata(noisy_signal)
    filtered_line.set_ydata(filtered_signal)

    noisy_line.set_visible(state["show_noise"])

    all_values = np.concatenate([clean_signal, noisy_signal, filtered_signal])
    ax.set_ylim(all_values.min() - 0.5, all_values.max() + 0.5)

    fig.canvas.draw_idle()


def reset(event):
    amplitude_slider.reset()
    frequency_slider.reset()
    phase_slider.reset()
    noise_mean_slider.reset()
    noise_covariance_slider.reset()
    cutoff_slider.reset()

    state["noise"] = generate_noise(
        size=len(t),
        noise_mean=initial_noise_mean,
        noise_covariance=initial_noise_covariance,
        seed=42
    )
    state["noise_mean"] = initial_noise_mean
    state["noise_covariance"] = initial_noise_covariance
    state["show_noise"] = initial_show_noise

    noisy_line.set_visible(initial_show_noise)
    update_plot()


def toggle_noise(label):
    state["show_noise"] = not state["show_noise"]
    noisy_line.set_visible(state["show_noise"])
    fig.canvas.draw_idle()


amplitude_slider.on_changed(update_plot)
frequency_slider.on_changed(update_plot)
phase_slider.on_changed(update_plot)
noise_mean_slider.on_changed(update_plot)
noise_covariance_slider.on_changed(update_plot)
cutoff_slider.on_changed(update_plot)
reset_button.on_clicked(reset)
show_noise_checkbox.on_clicked(toggle_noise)

plt.show()


## 6. Порівняння чистої, шумної та відфільтрованої гармоніки

На графіку відображаються три сигнали:
- чиста гармоніка;
- гармоніка з шумом;
- відфільтрована гармоніка.

Фільтрація виконується низькочастотним фільтром Butterworth за допомогою `scipy.signal.filtfilt`.


In [ ]:
example_amplitude = 1.0
example_frequency = 0.5
example_phase = 0.0
example_noise_mean = 0.0
example_noise_covariance = 0.2
example_cutoff_frequency = 1.0

example_clean = harmonic(t, example_amplitude, example_frequency, example_phase)
example_noise = generate_noise(len(t), example_noise_mean, example_noise_covariance, seed=7)
example_noisy = add_noise(example_clean, example_noise)
example_filtered = filter_signal(example_noisy, example_cutoff_frequency, sampling_frequency)

plt.figure(figsize=(10, 6))
plt.plot(t, example_clean, linewidth=2, label="Clean harmonic")
plt.plot(t, example_noisy, alpha=0.5, label="Noisy harmonic")
plt.plot(t, example_filtered, linewidth=2, label="Filtered harmonic")
plt.title("Порівняння чистої, шумної та відфільтрованої гармоніки")
plt.xlabel("t")
plt.ylabel("y(t)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 7. Інструкція для користувача

1. Запустіть усі клітинки ноутбука.
2. У головному інтерактивному графіку змінюйте параметри за допомогою слайдерів.
3. Слайдер `Amplitude` змінює амплітуду гармоніки.
4. Слайдер `Frequency` змінює частоту гармоніки.
5. Слайдер `Phase` змінює фазовий зсув.
6. Слайдери `Noise Mean` та `Noise Covariance` змінюють параметри шуму.
7. Слайдер `Cutoff Frequency` змінює параметр фільтрації.
8. Checkbox `Show Noise` показує або приховує шумну гармоніку.
9. Кнопка `Reset` повертає всі параметри до початкових значень.

Якщо змінюються тільки параметри гармоніки, шум не генерується заново.  
Якщо змінюються параметри шуму, програма створює новий шум.


## Висновок

У лабораторній роботі створено інтерактивну програму для побудови гармонічної функції з шумом.  
Реалізовано слайдери для зміни параметрів гармоніки, шуму та фільтра.  
Також додано чекбокс для показу або приховування шуму та кнопку `Reset` для відновлення початкових значень.

Шумна гармоніка була відфільтрована за допомогою низькочастотного фільтра Butterworth, реалізованого через `scipy.signal.filtfilt`.
